# Benchmark d'optimisation — RetinaNet R50 & EfficientDet D4

Comparaison des techniques d'optimisation pour l'inférence GPU, organisées par
disponibilité (le notebook s'adapte automatiquement à l'environnement) :

| Technique | Ce qu'elle fait | Windows | Colab |
|---|---|:---:|:---:|
| **Baseline** | modèle PyTorch FP32 brut | ✓ | ✓ |
| **FP16 autocast** | calcul demi-précision sur Tensor Cores (sans compilation) | ✓ | ✓ |
| **TorchScript** | compilation de graphe + fusion Conv-BN-ReLU | ✓ | ✓ |
| **torch.compile** | fusion kernels (`inductor`/Triton) ou capture graphe CUDA (`cudagraphs`) | ✓ (cudagraphs) | ✓ (inductor) |
| **ONNX Runtime** | runtime cross-platform, CUDA EP | ✓ | ✓ |
| **TensorRT FP16** | fusion kernels + FP16 compilés en CUDA natif | ✗ | ✓ |
| **TensorRT INT8** | quantification entière + calibration COCO | ✗ | ✓ |

Pour chaque variante : **benchmark** (FPS + détail par module) + **évaluation MAP** COCO.
Les modèles optimisés sont sauvegardés dans `outputs/models/`.

> **Décomposition du speedup pour le rapport** : en mesurant `baseline → FP16 → TorchScript → TRT FP16`,
> on isole la contribution de chaque optimisation (gain FP16 pur vs gain fusion/graphe).

## 0. Installation des dépendances

### Sur Colab (Linux + GPU NVIDIA)
```bash
!pip install torch-tensorrt onnx onnxruntime-gpu onnxsim --quiet
```
TensorRT et CUDA sont pré-installés dans le runtime Colab GPU — seul `torch-tensorrt` est nécessaire.

### Sur Windows (développement local)
```bash
pip install onnx onnxruntime-gpu onnxsim
```
`torch-tensorrt` n'est **pas** installé sur Windows — TRT sera automatiquement désactivé.  
Les sections TRT se skippent proprement grâce à la détection d'environnement ci-dessous.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import gc
from pathlib import Path

import torch
import pandas as pd
import numpy as np

from utils.benchmark   import benchmark_model, ModuleBenchmark
from utils.data_loader import load_profiling_data, load_eval_data
import optimizations as opt

# ── Détection d'environnement (capability.py centralise tout) ─────────────────
CAPS            = opt.print_report()
TRT_AVAILABLE   = CAPS.flags["tensorrt"]
ORT_AVAILABLE   = CAPS.flags["ort"]
COMPILE_BACKEND = CAPS.compile_backend     # "inductor" | "cudagraphs" | "eager"

# ── Config ────────────────────────────────────────────────────────────────────
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
N_WARMUP  = 50
N_MEASURE = 500
BENCH_TAG = "optimization_v1"

IMG_DIR  = "datasets/coco/val2017"
ANN_FILE = "datasets/coco/annotations/instances_val2017.json"
N_TOTAL  = N_WARMUP + N_MEASURE
N_EVAL   = 500

# Répertoires de sortie
Path("outputs/models").mkdir(parents=True, exist_ok=True)
Path("outputs/onnx").mkdir(parents=True, exist_ok=True)
Path(f"results/benchmark/modules/{BENCH_TAG}").mkdir(parents=True, exist_ok=True)

if DEVICE == "cpu":
    print("\n⚠ Aucun GPU CUDA détecté — le benchmark et la plupart des optimisations")
    print("  nécessitent CUDA. Vérifier l'installation torch (build +cuXXX, pas +cpu).")

Plateforme   : Windows (local)
PyTorch      : 2.8.0+cu129
CUDA         : ✓ NVIDIA GeForce RTX 5060 Laptop GPU
Backend compile recommandé : cudagraphs

Techniques d'optimisation disponibles :
  FP16 autocast               ✓ disponible
  TorchScript                 ✓ disponible
  torch.compile (cudagraphs)  ✓ disponible
    └ inductor (Triton)       ✗ indisponible
  ONNX export                 ✓ disponible
  ONNX Runtime                ✓ disponible
  TensorRT FP16/INT8          ✗ indisponible


In [2]:
from pycocotools.coco import COCO

profile_data = load_profiling_data(IMG_DIR, ANN_FILE, n=N_TOTAL)

# load_eval_data retourne (LazySampleList, coco) — on unpacks
eval_data, _ = load_eval_data(IMG_DIR, ANN_FILE, n=N_EVAL)
coco_gt      = COCO(ANN_FILE)

print(f"Benchmark : {len(profile_data)} images")
print(f"Eval MAP  : {len(eval_data)} images")

loading annotations into memory...
Done (t=0.45s)
creating index...
index created!
loading annotations into memory...
Done (t=0.41s)
creating index...
index created!
loading annotations into memory...
Done (t=0.42s)
creating index...
index created!
Benchmark : 550 images
Eval MAP  : 500 images


In [3]:
# ── Helpers ───────────────────────────────────────────────────────────────────
results_log = []   # accumule tous les résultats pour le tableau final

def vram(label=""):
    alloc    = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    free     = torch.cuda.mem_get_info()[0] / 1e9
    print(f"  VRAM [{label:<28}]  alloc={alloc:.2f} GB  reserv={reserved:.2f} GB  libre={free:.2f} GB")

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def run_benchmark(model, preprocess_fn, collate_fn, label, model_name, with_modules=True):
    """Lance benchmark + ModuleBenchmark, sauvegarde le CSV modules, retourne dict résultats."""
    mb = ModuleBenchmark() if with_modules else None
    result = benchmark_model(
        model, profile_data, preprocess_fn, collate_fn,
        n_warmup=N_WARMUP, n_measure=N_MEASURE,
        device=DEVICE, module_benchmark=mb,
    )
    print(f"  [{label}] {result['mean_ms']:.2f} ms ± {result['std_ms']:.2f}  |  {result['fps']:.1f} FPS")
    if mb is not None and "modules" in result:
        df_mod = result["modules"]
        csv_path = f"results/benchmark/modules/{BENCH_TAG}/{model_name}_{label.replace(' ', '_')}.csv"
        df_mod.to_csv(csv_path, index=False)
        print(f"  → Modules sauvegardés : {csv_path}  ({len(df_mod)} lignes)")
    return result

def run_eval(model, preprocess_eval_fn, collate_fn, postprocess_eval_fn, label):
    """Lance l'évaluation MAP COCO, retourne dict AP*."""
    from eval.map_eval import run_map_evaluation
    print(f"  [{label}] Évaluation MAP ({N_EVAL} images)...")
    maps = run_map_evaluation(
        model, eval_data, coco_gt,
        preprocess_eval_fn, collate_fn, postprocess_eval_fn,
        device=DEVICE,
    )
    print(f"  AP={maps['AP']:.3f}  AP50={maps['AP50']:.3f}  AP75={maps['AP75']:.3f}")
    return maps

def record(model_name, label, bench, maps):
    results_log.append({
        "modèle":    model_name,
        "variante":  label,
        "mean_ms":   f"{bench['mean_ms']:.2f}",
        "FPS":       f"{bench['fps']:.1f}",
        "AP":        f"{maps['AP']:.3f}" if maps else "—",
        "AP50":      f"{maps['AP50']:.3f}" if maps else "—",
    })

print("Helpers prêts.")

Helpers prêts.


---
## 1. RetinaNet R50

Backbone ResNet-50 + FPN. Entrée : `List[Tensor[3,640,640]]`.

### 1.1 Baseline

In [4]:
import models.retinanet_r50 as r50

print("── RetinaNet R50 — Baseline ──────────────────────────────────")
model_r50 = r50.load_model(DEVICE)
vram("après load_model")

bench_r50_base = run_benchmark(model_r50, r50.preprocess, r50.collate, "baseline", "r50")
vram("après benchmark")

print("\n  Top 10 modules les plus lents :")
df_mod = bench_r50_base.get("modules", pd.DataFrame())
if not df_mod.empty:
    display(df_mod.head(10)[["module", "type", "mean_ms", "pct_sum"]])

map_r50_base = run_eval(model_r50, r50.preprocess_eval, r50.collate, r50.postprocess_eval, "baseline")
record("RetinaNet R50", "baseline", bench_r50_base, map_r50_base)

del model_r50; free_gpu()
vram("après cleanup")

── RetinaNet R50 — Baseline ──────────────────────────────────
  VRAM [après load_model            ]  alloc=0.16 GB  reserv=0.17 GB  libre=7.12 GB
  [baseline] 66.33 ms ± 4.67  |  15.1 FPS
  → Modules sauvegardés : results/benchmark/modules/optimization_v1/r50_baseline.csv  (160 lignes)
  VRAM [après benchmark             ]  alloc=0.16 GB  reserv=0.17 GB  libre=7.10 GB

  Top 10 modules les plus lents :


,module,type,mean_ms,pct_sum
0,backbone.fpn.layer_blocks.0.0,0,1.389393,5.72
1,backbone.body.conv1,conv1,0.890472,3.66
2,backbone.body.layer2.0.downsample.0,0,0.668202,2.75
3,backbone.body.layer4.0.downsample.0,0,0.663212,2.73
4,transform,transform,0.649739,2.67
5,backbone.body.layer3.0.downsample.0,0,0.631584,2.60
6,backbone.body.layer4.0.conv2,conv2,0.608402,2.50
7,backbone.body.layer4.2.conv2,conv2,0.551135,2.27
8,backbone.body.layer4.1.conv2,conv2,0.548801,2.26
9,anchor_generator,anchor_generator,0.479068,1.97


  [baseline] Évaluation MAP (500 images)...
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.85s).
Accumulating evaluation results...
DONE (t=0.90s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.413
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.604
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.443
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.189
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.470
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.633
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.348
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.519
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.553
 Average Recall     (AR)

### 1.2 FP16 autocast

Calcul en demi-précision sur les Tensor Cores, sans compilation ni calibration.
Les poids restent FP32, seules les conv/matmul passent en FP16. Isole le gain
**FP16 pur** (à comparer ensuite avec TRT FP16 pour mesurer l'apport de la fusion).

In [5]:
from optimizations import to_fp16_autocast

if DEVICE == "cpu":
    print("⚠ FP16 nécessite CUDA — section skippée.")
else:
    print("── RetinaNet R50 — FP16 autocast ────────────────────────────")
    model_r50      = r50.load_model(DEVICE)
    model_r50_fp16 = to_fp16_autocast(model_r50)
    vram("après wrap FP16")

    # with_modules=True : on garde le détail par module pour voir quelles couches
    # accélèrent le plus en FP16 (utile pour la décomposition du rapport)
    bench_r50_fp16 = run_benchmark(
        model_r50_fp16, r50.preprocess, r50.collate, "fp16", "r50", with_modules=True,
    )
    speedup = bench_r50_base["mean_ms"] / bench_r50_fp16["mean_ms"]
    print(f"  Speedup vs baseline : ×{speedup:.2f}")

    map_r50_fp16 = run_eval(model_r50_fp16, r50.preprocess_eval, r50.collate, r50.postprocess_eval, "fp16")
    record("RetinaNet R50", "FP16 autocast", bench_r50_fp16, map_r50_fp16)

    del model_r50, model_r50_fp16; free_gpu()
    vram("après cleanup")

── RetinaNet R50 — FP16 autocast ────────────────────────────
[FP16] Wrapper autocast créé (dtype=torch.float16).
  Poids conservés en FP32, calcul conv/matmul en FP16 sur Tensor Cores.
  VRAM [après wrap FP16             ]  alloc=0.16 GB  reserv=0.17 GB  libre=7.10 GB
  [fp16] 47.08 ms ± 4.21  |  21.2 FPS
  → Modules sauvegardés : results/benchmark/modules/optimization_v1/r50_fp16.csv  (160 lignes)
  Speedup vs baseline : ×1.41
  [fp16] Évaluation MAP (500 images)...
Loading and preparing results...
DONE (t=0.06s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.86s).
Accumulating evaluation results...
DONE (t=1.00s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.413
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.604
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.445
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0

### 1.3 TorchScript (compilation de graphe)

`torch.jit.script` → `freeze` → `optimize_for_inference`. Fusionne Conv+BatchNorm
(les stats BN figées sont repliées dans la conv) et Conv+ReLU. Fonctionne sur
Windows comme Colab, sans dépendance externe.

In [6]:
from optimizations import optimize_with_torchscript, save_torchscript

print("── RetinaNet R50 — TorchScript ──────────────────────────────")
model_r50 = r50.load_model(DEVICE)
example   = r50.collate([r50.preprocess(profile_data[0])], DEVICE)

# torchvision est scriptable → prefer="script" (gère la résolution native en eval)
model_r50_ts = optimize_with_torchscript(model_r50, example_input=example, prefer="script")
vram("après torchscript")

# with_modules=False : le graphe figé n'expose plus les modules feuilles aux hooks
bench_r50_ts = run_benchmark(
    model_r50_ts, r50.preprocess, r50.collate, "torchscript", "r50", with_modules=False,
)
speedup = bench_r50_base["mean_ms"] / bench_r50_ts["mean_ms"]
print(f"  Speedup vs baseline : ×{speedup:.2f}")

map_r50_ts = run_eval(model_r50_ts, r50.preprocess_eval, r50.collate, r50.postprocess_eval, "torchscript")
record("RetinaNet R50", "TorchScript", bench_r50_ts, map_r50_ts)

try:
    save_torchscript(model_r50_ts, "outputs/models/retinanet_r50_torchscript.ts")
except Exception as e:
    print(f"  Sauvegarde TorchScript : {e}")

del model_r50, model_r50_ts; free_gpu()
vram("après cleanup")

── RetinaNet R50 — TorchScript ──────────────────────────────
[TorchScript] ✓ Graphe capturé via torch.jit.script
[TorchScript] ✓ Freeze appliqué (poids inlinés, constant folding)
[TorchScript] ✓ optimize_for_inference (fusion Conv+BN+ReLU)
  → Le premier appel forward finalise les optimisations JIT (warmup).
  VRAM [après torchscript           ]  alloc=0.26 GB  reserv=0.27 GB  libre=6.99 GB


c:\ProgramData\anaconda3\Lib\site-packages\torchvision\models\detection\retinanet.py:672: UserWarning: RetinaNet always returns a (Losses, Detections) tuple in scripting
  warnings.warn("RetinaNet always returns a (Losses, Detections) tuple in scripting")


  [torchscript] 92.33 ms ± 43.62  |  10.8 FPS
  Speedup vs baseline : ×0.72
  [torchscript] Évaluation MAP (500 images)...


AttributeError: 'list' object has no attribute 'cpu'

### 1.4 torch.compile

Backend auto-sélectionné : **`inductor`** (Triton, Colab) ou **`cudagraphs`** (Windows, sans Triton).
- `inductor` : génère des kernels Triton fusionnés.
- `cudagraphs` : capture le graphe CUDA, élimine l'overhead de lancement des kernels.

In [7]:
from optimizations import compile_model, save_compiled

if DEVICE == "cpu":
    print("⚠ torch.compile sans CUDA n'apporte pas de gain — section skippée.")
else:
    print(f"── RetinaNet R50 — torch.compile (backend={COMPILE_BACKEND}) ──")
    model_r50          = r50.load_model(DEVICE)
    model_r50_compiled = compile_model(model_r50, backend=COMPILE_BACKEND)
    vram("après compile")

    # Le premier appel (warmup) déclenche la compilation JIT
    bench_r50_compiled = run_benchmark(
        model_r50_compiled, r50.preprocess, r50.collate,
        f"compile_{COMPILE_BACKEND}", "r50", with_modules=False,
    )
    vram("après benchmark")

    speedup = bench_r50_base["mean_ms"] / bench_r50_compiled["mean_ms"]
    print(f"  Speedup vs baseline : ×{speedup:.2f}")

    map_r50_compiled = run_eval(
        model_r50_compiled, r50.preprocess_eval, r50.collate, r50.postprocess_eval,
        f"compile_{COMPILE_BACKEND}",
    )
    record("RetinaNet R50", f"torch.compile ({COMPILE_BACKEND})", bench_r50_compiled, map_r50_compiled)

    save_compiled(model_r50_compiled, "outputs/models/retinanet_r50_compiled.pt")

    del model_r50, model_r50_compiled; free_gpu()
    vram("après cleanup")

── RetinaNet R50 — torch.compile (backend=cudagraphs) ──
[torch.compile] backend=cudagraphs
  Capture du graphe CUDA — élimine l'overhead de lancement des kernels.
  Aucune dépendance Triton. Shapes d'entrée supposées fixes (640×640).
  → Inclure le premier appel dans n_warmup (≥ 5 itérations de marge).
  VRAM [après compile               ]  alloc=0.34 GB  reserv=1.55 GB  libre=5.71 GB


skipping cudagraphs due to skipping cudagraphs due to cpu device (arg310_1). Found from : 
   File "c:\ProgramData\anaconda3\Lib\site-packages\torchvision\models\detection\retinanet.py", line 640, in forward
    anchors = self.anchor_generator(images, features)
  File "c:\ProgramData\anaconda3\Lib\site-packages\torchvision\models\detection\anchor_utils.py", line 126, in forward
    self.set_cell_anchors(dtype, device)
  File "c:\ProgramData\anaconda3\Lib\site-packages\torchvision\models\detection\anchor_utils.py", line 77, in set_cell_anchors
    self.cell_anchors = [cell_anchor.to(dtype=dtype, device=device) for cell_anchor in self.cell_anchors]

c:\ProgramData\anaconda3\Lib\site-packages\torchvision\_meta_registrations.py:173: FutureWarning: `create_unbacked_symint` is deprecated, please use `new_dynamic_size` instead
  num_to_keep = ctx.create_unbacked_symint()
skipping cudagraphs due to skipping cudagraphs due to multiple devices: 
c:\ProgramData\anaconda3\Lib\site-packages\torchvi

RuntimeError: The size of tensor a (1000) must match the size of tensor b (201) at non-singleton dimension 0

### 1.5 TensorRT FP16 *(Colab/Linux uniquement)*

In [8]:
if not TRT_AVAILABLE:
    print("⚠ TensorRT non disponible sur cet environnement.")
    print("  → Exécuter cette cellule sur Colab (Runtime GPU).")
    print("  → Sur Colab : !pip install torch-tensorrt --quiet")
else:
    from optimizations.tensorrt_fp16 import build_trt_fp16

    print("── RetinaNet R50 — TensorRT FP16 ───────────────────────────")
    model_r50     = r50.load_model(DEVICE)
    model_r50_trt = build_trt_fp16(model_r50, min_block_size=5)
    vram("après build_trt_fp16")

    bench_r50_trt = run_benchmark(
        model_r50_trt, r50.preprocess, r50.collate, "trt_fp16", "r50",
        with_modules=False,
    )
    vram("après benchmark")

    speedup_trt = bench_r50_base["mean_ms"] / bench_r50_trt["mean_ms"]
    print(f"  Speedup vs baseline : ×{speedup_trt:.2f}")

    map_r50_trt = run_eval(
        model_r50_trt, r50.preprocess_eval, r50.collate, r50.postprocess_eval, "trt_fp16"
    )
    record("RetinaNet R50", "TRT FP16", bench_r50_trt, map_r50_trt)

    try:
        from optimizations.tensorrt_fp16 import save_trt_model
        sample = r50.collate([r50.preprocess(profile_data[0])], DEVICE)
        save_trt_model(
            r50.load_model(DEVICE),
            "outputs/models/retinanet_r50_trt_fp16.ts",
            sample_input=sample[0].unsqueeze(0),
        )
    except Exception as e:
        print(f"  Sauvegarde engine persistant : {e}")

    del model_r50, model_r50_trt; free_gpu()
    vram("après cleanup")

⚠ TensorRT non disponible sur cet environnement.
  → Exécuter cette cellule sur Colab (Runtime GPU).
  → Sur Colab : !pip install torch-tensorrt --quiet


### 1.6 TensorRT INT8 *(Colab — calibration ~5 min, optionnel)*

In [9]:
if not TRT_AVAILABLE:
    print("⚠ TensorRT non disponible — section INT8 skippée.")
else:
    pass
    # Décommenter pour activer la quantification INT8 (~5 min de calibration)
    # ─────────────────────────────────────────────────────────────────────────
    # from optimizations.tensorrt_int8 import build_trt_int8, build_calibration_loader
    #
    # model_r50    = r50.load_model(DEVICE)
    # calib_loader = build_calibration_loader(profile_data, r50.preprocess, n_images=300, batch_size=8)
    # model_r50_int8 = build_trt_int8(
    #     model_r50, calib_loader,
    #     save_path="outputs/models/retinanet_r50_trt_int8.ts",
    # )
    # bench_r50_int8 = run_benchmark(model_r50_int8, r50.preprocess, r50.collate, "trt_int8", "r50", with_modules=False)
    # map_r50_int8   = run_eval(model_r50_int8, r50.preprocess_eval, r50.collate, r50.postprocess_eval, "trt_int8")
    # record("RetinaNet R50", "TRT INT8", bench_r50_int8, map_r50_int8)
    # del model_r50, model_r50_int8; free_gpu()

⚠ TensorRT non disponible — section INT8 skippée.


### 1.7 ONNX Runtime (backbone)

In [10]:
if not ORT_AVAILABLE:
    print("⚠ onnxruntime-gpu non installé.")
    print("  pip install onnxruntime-gpu")
else:
    from optimizations.onnx_export import export_backbone_only, check_onnx
    from optimizations.ort_inference import build_ort_model

    print("── RetinaNet R50 — ONNX Runtime ────────────────────────────")
    model_r50 = r50.load_model(DEVICE)
    onnx_path = "outputs/onnx/retinanet_r50_backbone.onnx"

    export_backbone_only(model_r50, onnx_path, image_size=(640, 640), device=DEVICE, simplify=True)
    check_onnx(onnx_path)
    vram("après export ONNX")

    ort_r50 = build_ort_model(onnx_path, device=DEVICE)

    bench_r50_ort = run_benchmark(
        ort_r50, r50.preprocess, r50.collate, "ort_backbone", "r50",
        with_modules=False,
    )
    vram("après benchmark ORT")

    speedup_ort = bench_r50_base["mean_ms"] / bench_r50_ort["mean_ms"]
    print(f"  Speedup backbone vs baseline : ×{speedup_ort:.2f}")
    print("  (backbone seulement — MAP non comparable directement)")

    record("RetinaNet R50", "ORT backbone", bench_r50_ort, None)

    del model_r50; free_gpu()
    vram("après cleanup")

── RetinaNet R50 — ONNX Runtime ────────────────────────────
[ONNX] Graphe simplifié avec onnxsim ✓
[ONNX] backbone exporté → outputs/onnx/retinanet_r50_backbone.onnx  (126.0 MB)
  outputs : ['feat_0', 'feat_1', 'feat_2', 'feat_3', 'feat_4']
[ONNX] ✓ Validation OK — outputs/onnx/retinanet_r50_backbone.onnx
  VRAM [après export ONNX           ]  alloc=0.51 GB  reserv=1.35 GB  libre=5.88 GB
[ORT] Session créée — provider actif : CUDAExecutionProvider
  inputs  : ['images']
  outputs : ['feat_0', 'feat_1', 'feat_2', 'feat_3', 'feat_4']
  [ort_backbone] 29.48 ms ± 1.51  |  33.9 FPS
  VRAM [après benchmark ORT         ]  alloc=0.48 GB  reserv=0.87 GB  libre=5.80 GB
  Speedup backbone vs baseline : ×2.25
  (backbone seulement — MAP non comparable directement)
  VRAM [après cleanup               ]  alloc=0.33 GB  reserv=0.86 GB  libre=5.82 GB


### 1.8 Comparaison des modules avant/après optimisation

Avant : Conv2d, BatchNorm2d, ReLU sont des modules distincts mesurables.
Après TorchScript/TRT : fusionnés (Conv-BN repliés, CBR kernel) → ils disparaissent du profil.

In [11]:
# Charger les CSV baseline et les afficher côte à côte
import os

baseline_csv = f"results/benchmark/modules/{BENCH_TAG}/r50_baseline.csv"
if os.path.exists(baseline_csv):
    df_base = pd.read_csv(baseline_csv)

    print("Top 15 modules — Baseline (avant optimisation) :")
    display(df_base.head(15)[["module", "type", "mean_ms", "std_ms", "pct_sum"]])

    print("\nTemps moyen par type de couche :")
    by_type = df_base.groupby("type")["mean_ms"].agg(["sum", "mean", "count"])
    by_type.columns = ["total_ms", "mean_ms", "n_modules"]
    display(by_type.sort_values("total_ms", ascending=False))

    print("\nRépartition par composant racine (backbone / fpn / head) :")
    by_root = df_base.groupby("root_component")["mean_ms"].sum().sort_values(ascending=False)
    display(by_root.reset_index())
else:
    print(f"CSV non trouvé : {baseline_csv} — exécuter la cellule baseline d'abord")

Top 15 modules — Baseline (avant optimisation) :


,module,type,mean_ms,std_ms,pct_sum
0,backbone.fpn.layer_blocks.0.0,0,1.389393,0.039425,5.72
1,backbone.body.conv1,conv1,0.890472,0.046973,3.66
2,backbone.body.layer2.0.downsample.0,0,0.668202,0.027620,2.75
3,backbone.body.layer4.0.downsample.0,0,0.663212,0.019307,2.73
4,transform,transform,0.649739,0.289849,2.67
5,backbone.body.layer3.0.downsample.0,0,0.631584,0.031348,2.60
6,backbone.body.layer4.0.conv2,conv2,0.608402,0.018639,2.50
7,backbone.body.layer4.2.conv2,conv2,0.551135,0.015010,2.27
8,backbone.body.layer4.1.conv2,conv2,0.548801,0.011344,2.26
9,anchor_generator,anchor_generator,0.479068,0.213252,1.97



Temps moyen par type de couche :


,total_ms,mean_ms,n_modules
type,,,
conv2,6.348185,0.396762,16
0,6.250151,0.347231,18
conv1,3.438219,0.202248,17
conv3,2.553583,0.159599,16
bn3,1.138565,0.071160,16
relu,0.794718,0.046748,17
transform,0.649739,0.649739,1
bn1,0.624604,0.036741,17
anchor_generator,0.479068,0.479068,1



Répartition par composant racine (backbone / fpn / head) :


,root_component,mean_ms
0,backbone,20.938844
1,head,2.233304
2,transform,0.649739
3,anchor_generator,0.479068


### 1.9 Résumé RetinaNet R50

In [12]:
df_r50 = pd.DataFrame([r for r in results_log if r["modèle"] == "RetinaNet R50"])
print("RetinaNet R50 — Résultats :")
display(df_r50)

RetinaNet R50 — Résultats :


,modèle,variante,mean_ms,FPS,AP,AP50
0,RetinaNet R50,baseline,66.33,15.1,0.413,0.604
1,RetinaNet R50,FP16 autocast,47.08,21.2,0.413,0.604
2,RetinaNet R50,ORT backbone,29.48,33.9,—,—


---
## 2. EfficientDet D4

Backbone EfficientNet-B4 + BiFPN. Entrée : `Tensor[B, 3, 640, 640]`.

> EfficientDet a une API différente de RetinaNet :
> - `collate` retourne un `Tensor[B,C,H,W]` (pas une liste)
> - `preprocess` retourne un `Tensor[1,3,H,W]` (avec dimension batch)

### 2.1 Baseline

In [13]:
import models.efficientdet_d4 as d4

print("── EfficientDet D4 — Baseline ────────────────────────────────")
model_d4 = d4.load_model(DEVICE)
vram("après load_model")

bench_d4_base = run_benchmark(model_d4, d4.preprocess, d4.collate, "baseline", "d4")
vram("après benchmark")

print("\n  Top 10 modules les plus lents :")
df_mod_d4 = bench_d4_base.get("modules", pd.DataFrame())
if not df_mod_d4.empty:
    display(df_mod_d4.head(10)[["module", "type", "mean_ms", "pct_sum"]])

map_d4_base = run_eval(model_d4, d4.preprocess_eval, d4.collate, d4.postprocess_eval, "baseline")
record("EfficientDet D4", "baseline", bench_d4_base, map_d4_base)

del model_d4; free_gpu()
vram("après cleanup")

── EfficientDet D4 — Baseline ────────────────────────────────
  VRAM [après load_model            ]  alloc=0.42 GB  reserv=0.90 GB  libre=5.78 GB
  [baseline] 116.34 ms ± 15.86  |  8.6 FPS
  → Modules sauvegardés : results/benchmark/modules/optimization_v1/d4_baseline.csv  (898 lignes)
  VRAM [après benchmark             ]  alloc=0.42 GB  reserv=0.90 GB  libre=5.78 GB

  Top 10 modules les plus lents :


,module,type,mean_ms,pct_sum
0,model.backbone.blocks.1.0.conv_dw,conv_dw,1.082754,1.88
1,model.backbone.conv_stem,conv_stem,0.530321,0.92
2,model.backbone.blocks.0.0.conv_dw,conv_dw,0.524162,0.91
3,model.backbone.blocks.1.1.conv_dw,conv_dw,0.502072,0.87
4,model.backbone.blocks.1.2.conv_dw,conv_dw,0.502036,0.87
5,model.backbone.blocks.1.3.conv_dw,conv_dw,0.501505,0.87
6,model.backbone.blocks.2.0.conv_dw,conv_dw,0.434892,0.76
7,model.backbone.blocks.1.0.bn1.act,act,0.406819,0.71
8,model.backbone.blocks.1.0.conv_pw,conv_pw,0.397071,0.69
9,model.backbone.blocks.2.3.conv_dw,conv_dw,0.378929,0.66


  [baseline] Évaluation MAP (500 images)...


RuntimeError: stack expects each tensor to be equal size, but got [7, 224, 16, 16] at entry 0 and [7, 224, 10, 10] at entry 1

### 2.2 FP16 autocast

In [14]:
from optimizations import to_fp16_autocast

if DEVICE == "cpu":
    print("⚠ FP16 nécessite CUDA — section skippée.")
else:
    print("── EfficientDet D4 — FP16 autocast ──────────────────────────")
    model_d4      = d4.load_model(DEVICE)
    model_d4_fp16 = to_fp16_autocast(model_d4)
    vram("après wrap FP16")

    bench_d4_fp16 = run_benchmark(
        model_d4_fp16, d4.preprocess, d4.collate, "fp16", "d4", with_modules=True,
    )
    speedup = bench_d4_base["mean_ms"] / bench_d4_fp16["mean_ms"]
    print(f"  Speedup vs baseline : ×{speedup:.2f}")

    map_d4_fp16 = run_eval(model_d4_fp16, d4.preprocess_eval, d4.collate, d4.postprocess_eval, "fp16")
    record("EfficientDet D4", "FP16 autocast", bench_d4_fp16, map_d4_fp16)

    del model_d4, model_d4_fp16; free_gpu()
    vram("après cleanup")

── EfficientDet D4 — FP16 autocast ──────────────────────────
[FP16] Wrapper autocast créé (dtype=torch.float16).
  Poids conservés en FP32, calcul conv/matmul en FP16 sur Tensor Cores.
  VRAM [après wrap FP16             ]  alloc=0.65 GB  reserv=5.35 GB  libre=0.48 GB
  [fp16] 128.03 ms ± 9.64  |  7.8 FPS
  → Modules sauvegardés : results/benchmark/modules/optimization_v1/d4_fp16.csv  (898 lignes)
  Speedup vs baseline : ×0.91
  [fp16] Évaluation MAP (500 images)...


RuntimeError: stack expects each tensor to be equal size, but got [6, 224, 16, 16] at entry 0 and [6, 224, 10, 10] at entry 1

### 2.3 TorchScript (compilation de graphe)

> EfficientDet (`effdet`) n'est généralement pas *scriptable* → on utilise le
> *tracing* à résolution figée (640×640). La MAP n'est donc pas évaluée ici
> (la trace ne supporte pas la résolution native 1024 de l'eval). Benchmark seul.

In [15]:
from optimizations import optimize_with_torchscript

print("── EfficientDet D4 — TorchScript (benchmark uniquement) ─────")
model_d4 = d4.load_model(DEVICE)
example  = d4.collate([d4.preprocess(profile_data[0])], DEVICE)

# effdet n'est généralement pas scriptable → trace figée à 640×640.
# La trace bake la résolution → on ne mesure que le benchmark (pas la MAP native).
model_d4_ts = optimize_with_torchscript(model_d4, example_input=example, prefer="trace")
vram("après torchscript")

bench_d4_ts = run_benchmark(
    model_d4_ts, d4.preprocess, d4.collate, "torchscript", "d4", with_modules=False,
)
speedup = bench_d4_base["mean_ms"] / bench_d4_ts["mean_ms"]
print(f"  Speedup vs baseline : ×{speedup:.2f}")
print("  Note : trace figée à 640×640 → MAP non évaluée (résolution native incompatible).")

record("EfficientDet D4", "TorchScript", bench_d4_ts, None)

del model_d4, model_d4_ts; free_gpu()
vram("après cleanup")

── EfficientDet D4 — TorchScript (benchmark uniquement) ─────
[TorchScript] ✓ Graphe capturé via torch.jit.trace
[TorchScript] ✓ Freeze appliqué (poids inlinés, constant folding)
[TorchScript] ✓ optimize_for_inference (fusion Conv+BN+ReLU)
  → Le premier appel forward finalise les optimisations JIT (warmup).
  VRAM [après torchscript           ]  alloc=0.90 GB  reserv=3.97 GB  libre=2.71 GB
  [torchscript] 43.52 ms ± 4.73  |  23.0 FPS
  Speedup vs baseline : ×2.67
  Note : trace figée à 640×640 → MAP non évaluée (résolution native incompatible).
  VRAM [après cleanup               ]  alloc=0.60 GB  reserv=1.20 GB  libre=5.48 GB


### 2.4 torch.compile

In [16]:
if DEVICE == "cpu":
    print("⚠ torch.compile sans CUDA n'apporte pas de gain — section skippée.")
else:
    print(f"── EfficientDet D4 — torch.compile (backend={COMPILE_BACKEND}) ──")
    model_d4          = d4.load_model(DEVICE)
    model_d4_compiled = compile_model(model_d4, backend=COMPILE_BACKEND)
    vram("après compile")

    bench_d4_compiled = run_benchmark(
        model_d4_compiled, d4.preprocess, d4.collate,
        f"compile_{COMPILE_BACKEND}", "d4", with_modules=False,
    )

    speedup = bench_d4_base["mean_ms"] / bench_d4_compiled["mean_ms"]
    print(f"  Speedup vs baseline : ×{speedup:.2f}")

    map_d4_compiled = run_eval(
        model_d4_compiled, d4.preprocess_eval, d4.collate, d4.postprocess_eval,
        f"compile_{COMPILE_BACKEND}",
    )
    record("EfficientDet D4", f"torch.compile ({COMPILE_BACKEND})", bench_d4_compiled, map_d4_compiled)

    save_compiled(model_d4_compiled, "outputs/models/efficientdet_d4_compiled.pt")

    del model_d4, model_d4_compiled; free_gpu()
    vram("après cleanup")

── EfficientDet D4 — torch.compile (backend=cudagraphs) ──
[torch.compile] backend=cudagraphs
  Capture du graphe CUDA — élimine l'overhead de lancement des kernels.
  Aucune dépendance Triton. Shapes d'entrée supposées fixes (640×640).
  → Inclure le premier appel dans n_warmup (≥ 5 itérations de marge).
  VRAM [après compile               ]  alloc=0.68 GB  reserv=1.20 GB  libre=5.47 GB


c:\ProgramData\anaconda3\Lib\site-packages\torchvision\_meta_registrations.py:173: FutureWarning: `create_unbacked_symint` is deprecated, please use `new_dynamic_size` instead
  num_to_keep = ctx.create_unbacked_symint()


  [compile_cudagraphs] 50.80 ms ± 1.36  |  19.7 FPS
  Speedup vs baseline : ×2.29
  [compile_cudagraphs] Évaluation MAP (500 images)...


KeyboardInterrupt: 

### 2.5 TensorRT FP16 *(Colab/Linux uniquement)*

> EfficientDet avec BiFPN contient des `swish` (SiLU) et des sommes pondérées — supporté depuis TensorRT 8.6+.

In [17]:
if not TRT_AVAILABLE:
    print("⚠ TensorRT non disponible sur cet environnement.")
    print("  → Exécuter cette cellule sur Colab (Runtime GPU).")
else:
    from optimizations.tensorrt_fp16 import build_trt_fp16

    print("── EfficientDet D4 — TensorRT FP16 ─────────────────────────")
    model_d4     = d4.load_model(DEVICE)
    model_d4_trt = build_trt_fp16(model_d4, min_block_size=5)
    vram("après build_trt_fp16")

    bench_d4_trt = run_benchmark(
        model_d4_trt, d4.preprocess, d4.collate, "trt_fp16", "d4",
        with_modules=False,
    )
    vram("après benchmark")

    speedup_trt = bench_d4_base["mean_ms"] / bench_d4_trt["mean_ms"]
    print(f"  Speedup vs baseline : ×{speedup_trt:.2f}")

    map_d4_trt = run_eval(
        model_d4_trt, d4.preprocess_eval, d4.collate, d4.postprocess_eval, "trt_fp16"
    )
    record("EfficientDet D4", "TRT FP16", bench_d4_trt, map_d4_trt)

    # EfficientDet prend Tensor[B,C,H,W] → export ONNX full plus simple
    try:
        from optimizations.onnx_export import export_full_detection
        export_full_detection(
            model_d4, "outputs/onnx/efficientdet_d4_full.onnx",
            image_size=(640, 640), device=DEVICE, simplify=True,
        )
    except Exception as e:
        print(f"  Export ONNX full : {e}")

    del model_d4, model_d4_trt; free_gpu()
    vram("après cleanup")

⚠ TensorRT non disponible sur cet environnement.
  → Exécuter cette cellule sur Colab (Runtime GPU).


### 2.6 Comparaison modules D4

In [18]:
baseline_csv_d4 = f"results/benchmark/modules/{BENCH_TAG}/d4_baseline.csv"
if os.path.exists(baseline_csv_d4):
    df_base_d4 = pd.read_csv(baseline_csv_d4)

    print("Top 15 modules — EfficientDet D4 Baseline :")
    display(df_base_d4.head(15)[["module", "type", "mean_ms", "std_ms", "pct_sum"]])

    print("\nRépartition par type :")
    by_type_d4 = df_base_d4.groupby("type")["mean_ms"].agg(["sum", "mean", "count"])
    by_type_d4.columns = ["total_ms", "mean_ms", "n_modules"]
    display(by_type_d4.sort_values("total_ms", ascending=False))
else:
    print(f"CSV non trouvé : {baseline_csv_d4}")

Top 15 modules — EfficientDet D4 Baseline :


,module,type,mean_ms,std_ms,pct_sum
0,model.backbone.blocks.1.0.conv_dw,conv_dw,1.082754,0.059871,1.88
1,model.backbone.conv_stem,conv_stem,0.530321,0.129987,0.92
2,model.backbone.blocks.0.0.conv_dw,conv_dw,0.524162,0.038364,0.91
3,model.backbone.blocks.1.1.conv_dw,conv_dw,0.502072,0.029964,0.87
4,model.backbone.blocks.1.2.conv_dw,conv_dw,0.502036,0.021948,0.87
5,model.backbone.blocks.1.3.conv_dw,conv_dw,0.501505,0.024061,0.87
6,model.backbone.blocks.2.0.conv_dw,conv_dw,0.434892,0.029507,0.76
7,model.backbone.blocks.1.0.bn1.act,act,0.406819,0.042299,0.71
8,model.backbone.blocks.1.0.conv_pw,conv_pw,0.397071,0.044176,0.69
9,model.backbone.blocks.2.3.conv_dw,conv_dw,0.378929,0.031029,0.66



Répartition par type :


,total_ms,mean_ms,n_modules
type,,,
conv_dw,14.038283,0.143248,98
conv_pw,11.948320,0.121922,98
bn,6.983516,0.068466,102
act,5.267122,0.034426,153
downsample,3.785875,0.126196,30
conv_pwl,3.726220,0.124207,30
conv_expand,2.210093,0.069065,32
conv_reduce,2.123609,0.066363,32
upsample,1.635469,0.058410,28


---
## 3. Tableau comparatif global

In [19]:
df_all = pd.DataFrame(results_log)
print("Résultats complets :")
display(df_all)

# Sauvegarder
df_all.to_csv(f"results/benchmark/modules/{BENCH_TAG}/comparaison_globale.csv", index=False)
print(f"\nSauvegardé → results/benchmark/modules/{BENCH_TAG}/comparaison_globale.csv")

Résultats complets :


,modèle,variante,mean_ms,FPS,AP,AP50
0,RetinaNet R50,baseline,66.33,15.1,0.413,0.604
1,RetinaNet R50,FP16 autocast,47.08,21.2,0.413,0.604
2,RetinaNet R50,ORT backbone,29.48,33.9,—,—
3,EfficientDet D4,TorchScript,43.52,23.0,—,—



Sauvegardé → results/benchmark/modules/optimization_v1/comparaison_globale.csv


In [20]:
# Speedup relatif par rapport au baseline de chaque modèle
rows = []
for model_name in df_all["modèle"].unique():
    subset = df_all[df_all["modèle"] == model_name].copy()
    base_ms = float(subset[subset["variante"] == "baseline"]["mean_ms"].values[0])
    for _, row in subset.iterrows():
        cur_ms  = float(row["mean_ms"])
        speedup = base_ms / cur_ms
        rows.append({
            "modèle":   model_name,
            "variante": row["variante"],
            "mean_ms":  row["mean_ms"],
            "FPS":      row["FPS"],
            "speedup":  f"×{speedup:.2f}",
            "AP":       row["AP"],
            "AP50":     row["AP50"],
        })

df_speedup = pd.DataFrame(rows)
print("Speedup par variante (relatif au baseline) :")
display(df_speedup)

IndexError: index 0 is out of bounds for axis 0 with size 0

## 4. Analyse — d'où viennent les gains ?

### Décomposition du speedup

En mesurant la chaîne `baseline → FP16 → TorchScript → TRT FP16`, on attribue
chaque tranche de gain à une cause précise :

| Transition | Cause du gain |
|---|---|
| baseline → **FP16 autocast** | demi-précision sur les Tensor Cores (conv/matmul 2× plus rapides), **sans** fusion |
| baseline → **TorchScript** | fusion de graphe (Conv+BN repliés, Conv+ReLU via NNC), reste en FP32 |
| FP16 → **TRT FP16** | apport de la **fusion de kernels CUDA** + optimisation du plan d'exécution, au-delà du seul FP16 |

→ Si `TRT FP16 ≫ FP16 autocast`, le gain vient surtout de la **fusion**.
→ Si `TRT FP16 ≈ FP16 autocast`, le modèle était déjà *bandwidth-bound* et le FP16 explique l'essentiel.

### Fusion Conv-BN-ReLU (CBR kernel)

Un bloc résiduel = 6+ opérations séquentielles (3 Conv, 3 BN, 2 ReLU). Chaque op
lit/écrit en VRAM → la bande passante mémoire est le goulot d'étranglement.

- **TorchScript** (`optimize_for_inference`) replie BatchNorm dans la conv précédente
  (en inférence, BN = transformation affine constante `y = γ·(x−μ)/σ + β`) → les
  modules `BatchNorm2d` disparaissent du graphe.
- **TensorRT** va plus loin : Conv+BN+ReLU → 1 seul kernel CUDA (CBR), 1 lecture
  des poids, 1 passe sur les activations, 1 écriture.

### FP16 Tensor Cores

Depuis Volta (V100), les Tensor Cores accélèrent les multiplications matricielles
FP16. Sur une T4 (Turing) : ~65 TFLOPS FP16 vs ~8 TFLOPS FP32.

### BiFPN vs FPN — différence d'accélérabilité

- **FPN** (RetinaNet) : additions simples entre niveaux → fusion triviale pour TRT.
- **BiFPN** (EfficientDet) : **sommes pondérées** (`wᵢ / (ε + Σwⱼ)`) → op non-standard,
  moins de fusion possible.

→ RetinaNet s'accélère typiquement mieux qu'EfficientDet à partir de TRT FP16.
C'est une conclusion **architecturale** transversale (point 2 du cahier des charges).

In [ ]:
# Visualiser la fusion : quels modules Conv/BN/ReLU disparaissent après TRT
# (démonstration qualitative)
if os.path.exists(baseline_csv):
    df_base = pd.read_csv(baseline_csv)

    conv_time  = df_base[df_base["type"].str.contains("Conv", na=False)]["mean_ms"].sum()
    bn_time    = df_base[df_base["type"].str.contains("BatchNorm", na=False)]["mean_ms"].sum()
    relu_time  = df_base[df_base["type"].str.contains("ReLU|SiLU", na=False)]["mean_ms"].sum()
    total_time = df_base["mean_ms"].sum()

    print("Décomposition baseline RetinaNet R50 :")
    print(f"  Conv  : {conv_time:.2f} ms  ({conv_time/total_time*100:.1f}%)")
    print(f"  BN    : {bn_time:.2f} ms  ({bn_time/total_time*100:.1f}%)")
    print(f"  ReLU  : {relu_time:.2f} ms  ({relu_time/total_time*100:.1f}%)")
    cbr_total = conv_time + bn_time + relu_time
    print(f"  Total CBR-fusionnables : {cbr_total:.2f} ms  ({cbr_total/total_time*100:.1f}%)")
    print()
    print("→ Après TRT FP16, ces opérations sont fusionnées en 1 kernel.")
    print(f"  Gain théorique sur CBR seul : ~×2 (FP16) × bande-passante réduite")
else:
    print("Exécuter la cellule baseline R50 d'abord.")